In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report


In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
data_path = 'C:\GaTech\MS-CSE\ISYE 6740\WiFOP\data\\final datasets\COMBINE_WIFOP_DATASET 1\COMBINE_WIFOP_DATASET\data\processed\ca_5km_daily_panel_2020_2023_6_9.csv'
data = pd.read_csv(data_path, sep=',')
data.head()

,date,grid_id,fire_today,lon,lat,w_lat,w_lon,T2M,PS,WS10M,...,pct_Pasture/Hay,pct_Cultivated Crops,pct_Woody Wetlands,pct_Emergent Herbaceous Wetlands,T2M_3d_mean,RH2M_3d_min,WS10M_3d_max,PRECTOT_7d_sum,fires_last_7d,fires_last_14d
0,2020-06-01,CA5K_00000,0,-124.350397,40.266353,40.5,-124.375,12.43,100.06,5.77,...,0.0,0.0,0.0,0.0,12.430000,90.21,5.77,0.08,NaN,NaN
1,2020-06-02,CA5K_00000,0,-124.350397,40.266353,40.5,-124.375,12.96,100.23,6.59,...,0.0,0.0,0.0,0.0,12.695000,89.99,6.59,0.08,0.0,0.0
2,2020-06-03,CA5K_00000,0,-124.350397,40.266353,40.5,-124.375,12.96,100.23,7.16,...,0.0,0.0,0.0,0.0,12.783333,89.99,7.16,0.08,0.0,0.0
3,2020-06-04,CA5K_00000,0,-124.350397,40.266353,40.5,-124.375,12.57,99.92,8.24,...,0.0,0.0,0.0,0.0,12.830000,89.65,8.24,0.08,0.0,0.0
4,2020-06-05,CA5K_00000,0,-124.350397,40.266353,40.5,-124.375,12.04,99.44,5.62,...,0.0,0.0,0.0,0.0,12.523333,89.65,8.24,0.18,0.0,0.0


In [4]:
data.isna().sum()

date                                     0
grid_id                                  0
fire_today                               0
lon                                      0
lat                                      0
w_lat                                    0
w_lon                                    0
T2M                                      0
PS                                       0
WS10M                                    0
PRECTOTCORR                              0
RH2M                                     0
n_lat                                    0
n_lon                                    0
ndvi                                510859
ndvi_interp                         400117
ndvi_missing_flag                   270416
ndvi_missing_streak                 270416
ndvi_7d_mean                        556669
ndvi_21d_mean                       888581
ndvi_drop                           897443
pct_Open Water                           0
pct_Perennial Ice/Snow                   0
pct_Develop

In [5]:
data.shape

(8247688, 43)

In [6]:
target = 'fire_today'
y = data.pop(target) #target label
data.head()

,date,grid_id,lon,lat,w_lat,w_lon,T2M,PS,WS10M,PRECTOTCORR,...,pct_Pasture/Hay,pct_Cultivated Crops,pct_Woody Wetlands,pct_Emergent Herbaceous Wetlands,T2M_3d_mean,RH2M_3d_min,WS10M_3d_max,PRECTOT_7d_sum,fires_last_7d,fires_last_14d
0,2020-06-01,CA5K_00000,-124.350397,40.266353,40.5,-124.375,12.43,100.06,5.77,0.08,...,0.0,0.0,0.0,0.0,12.430000,90.21,5.77,0.08,NaN,NaN
1,2020-06-02,CA5K_00000,-124.350397,40.266353,40.5,-124.375,12.96,100.23,6.59,0.00,...,0.0,0.0,0.0,0.0,12.695000,89.99,6.59,0.08,0.0,0.0
2,2020-06-03,CA5K_00000,-124.350397,40.266353,40.5,-124.375,12.96,100.23,7.16,0.00,...,0.0,0.0,0.0,0.0,12.783333,89.99,7.16,0.08,0.0,0.0
3,2020-06-04,CA5K_00000,-124.350397,40.266353,40.5,-124.375,12.57,99.92,8.24,0.00,...,0.0,0.0,0.0,0.0,12.830000,89.65,8.24,0.08,0.0,0.0
4,2020-06-05,CA5K_00000,-124.350397,40.266353,40.5,-124.375,12.04,99.44,5.62,0.10,...,0.0,0.0,0.0,0.0,12.523333,89.65,8.24,0.18,0.0,0.0


In [7]:
y.value_counts()

fire_today
0    8226135
1      21553
Name: count, dtype: int64

In [8]:
le = LabelEncoder()
y = le.fit_transform(y)

In [9]:
for col in data.columns:
    if col == 'grid_id' or col == 'date':
        continue
    data[col] = data[col].fillna(data[col].median())

In [10]:
data.head(2)

,date,grid_id,lon,lat,w_lat,w_lon,T2M,PS,WS10M,PRECTOTCORR,...,pct_Pasture/Hay,pct_Cultivated Crops,pct_Woody Wetlands,pct_Emergent Herbaceous Wetlands,T2M_3d_mean,RH2M_3d_min,WS10M_3d_max,PRECTOT_7d_sum,fires_last_7d,fires_last_14d
0,2020-06-01,CA5K_00000,-124.350397,40.266353,40.5,-124.375,12.43,100.06,5.77,0.08,...,0.0,0.0,0.0,0.0,12.430,90.21,5.77,0.08,0.0,0.0
1,2020-06-02,CA5K_00000,-124.350397,40.266353,40.5,-124.375,12.96,100.23,6.59,0.00,...,0.0,0.0,0.0,0.0,12.695,89.99,6.59,0.08,0.0,0.0


In [11]:
X = data.iloc[:, 2:]
X.head()

,lon,lat,w_lat,w_lon,T2M,PS,WS10M,PRECTOTCORR,RH2M,n_lat,...,pct_Pasture/Hay,pct_Cultivated Crops,pct_Woody Wetlands,pct_Emergent Herbaceous Wetlands,T2M_3d_mean,RH2M_3d_min,WS10M_3d_max,PRECTOT_7d_sum,fires_last_7d,fires_last_14d
0,-124.350397,40.266353,40.5,-124.375,12.43,100.06,5.77,0.08,90.21,40.274998,...,0.0,0.0,0.0,0.0,12.430000,90.21,5.77,0.08,0.0,0.0
1,-124.350397,40.266353,40.5,-124.375,12.96,100.23,6.59,0.00,89.99,40.274998,...,0.0,0.0,0.0,0.0,12.695000,89.99,6.59,0.08,0.0,0.0
2,-124.350397,40.266353,40.5,-124.375,12.96,100.23,7.16,0.00,90.62,40.274998,...,0.0,0.0,0.0,0.0,12.783333,89.99,7.16,0.08,0.0,0.0
3,-124.350397,40.266353,40.5,-124.375,12.57,99.92,8.24,0.00,89.65,40.274998,...,0.0,0.0,0.0,0.0,12.830000,89.65,8.24,0.08,0.0,0.0
4,-124.350397,40.266353,40.5,-124.375,12.04,99.44,5.62,0.10,91.41,40.274998,...,0.0,0.0,0.0,0.0,12.523333,89.65,8.24,0.18,0.0,0.0


In [ ]:
#smote = SMOTE()
#X_smoted, y_smoted = smote.fit_resample(X,y)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.20)

In [22]:
X_train.shape, X_test.shape

((6598150, 40), (1649538, 40))

In [18]:
class Model(tf.keras.Model):
    def __init__(self, input_dimension, neurons, num_classes):
        super().__init__()

        #Convolutional Layer
        self.conv1 = tf.keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')
        self.batch1 = tf.keras.layers.BatchNormalization()
        self.maxpool1 = tf.keras.layers.MaxPool1D(3)

        #Convolutional Layer
        self.conv2 = tf.keras.layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')
        self.batch2 = tf.keras.layers.BatchNormalization()
        self.maxpool2 = tf.keras.layers.MaxPool1D(3)

        #Flatten Layer
        self.flatten = tf.keras.layers.Flatten()

        #Dense Layer (hidden layer)
        self.dense = tf.keras.layers.Dense(units=neurons, activation='relu')
        self.batch3 = tf.keras.layers.BatchNormalization()
        self.output_layer = tf.keras.layers.Dense(units=num_classes, activation='sigmoid')

    def call(self, inputs):
        x = self.conv1(inputs)
        x = self.batch1(x)
        x = self.maxpool1(x)

        x = self.conv2(x)
        x = self.batch2(x)
        x = self.maxpool2(x)

        x = self.flatten(x)
        x = self.dense(x)
        x = self.batch3(x)
        output = self.output_layer(x)

        return output


In [24]:
n_features = X_train.shape[1]
model = Model(input_dimension=n_features, neurons=64, num_classes=1)
model.build(input_shape=(None, n_features, 1))
model.compile(loss=tf.keras.losses.BinaryCrossentropy(), optimizer='Adam', metrics=['accuracy'])

In [25]:
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_2 (Conv1D)           multiple                  256       
                                                                 
 batch_normalization_3 (Batc  multiple                 256       
 hNormalization)                                                 
                                                                 
 max_pooling1d_2 (MaxPooling  multiple                 0         
 1D)                                                             
                                                                 
 conv1d_3 (Conv1D)           multiple                  24704     
                                                                 
 batch_normalization_4 (Batc  multiple                 512       
 hNormalization)                                                 
                                                           

In [26]:
X_train_cnn = X_train.to_numpy().reshape(-1, n_features, 1)
X_test_cnn = X_test.to_numpy().reshape(-1, n_features, 1)

In [30]:
callbacks = [EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]

history = model.fit(X_train_cnn, y_train, batch_size=100, epochs=20, verbose=2, validation_split=0.1, callbacks=callbacks)

Epoch 1/20
59384/59384 - 223s - loss: 0.0152 - accuracy: 0.9974 - val_loss: 0.0161 - val_accuracy: 0.9973 - 223s/epoch - 4ms/step
Epoch 2/20
59384/59384 - 239s - loss: 0.0151 - accuracy: 0.9974 - val_loss: 0.0169 - val_accuracy: 0.9974 - 239s/epoch - 4ms/step
Epoch 3/20
59384/59384 - 222s - loss: 0.0151 - accuracy: 0.9974 - val_loss: 0.0157 - val_accuracy: 0.9974 - 222s/epoch - 4ms/step
Epoch 4/20
59384/59384 - 237s - loss: 0.0151 - accuracy: 0.9974 - val_loss: 0.0156 - val_accuracy: 0.9974 - 237s/epoch - 4ms/step
Epoch 5/20
59384/59384 - 226s - loss: 0.0150 - accuracy: 0.9974 - val_loss: 0.0153 - val_accuracy: 0.9973 - 226s/epoch - 4ms/step
Epoch 6/20
59384/59384 - 223s - loss: 0.0150 - accuracy: 0.9974 - val_loss: 0.0156 - val_accuracy: 0.9973 - 223s/epoch - 4ms/step
Epoch 7/20
59384/59384 - 239s - loss: 0.0150 - accuracy: 0.9974 - val_loss: 0.0173 - val_accuracy: 0.9974 - 239s/epoch - 4ms/step


In [31]:
predictions = model.predict(X_test_cnn)
y_pred = np.argmax(predictions, axis=1)

51549/51549 [==============================] - 59s 1ms/step


In [32]:
precision = precision_score(y_test, y_pred, zero_division=0, average='weighted')
recall = recall_score(y_test, y_pred, zero_division=0, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Precision score: {precision}, Recall score: {recall}, F1 score: {f1}")

Precision score: 0.9947799120856426, Recall score: 0.9973865409587411, F1 score: 0.9960815212143669


In [33]:
clr = classification_report(y_test, y_pred, zero_division=0)

In [34]:
print(clr)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1645227
           1       0.00      0.00      0.00      4311

    accuracy                           1.00   1649538
   macro avg       0.50      0.50      0.50   1649538
weighted avg       0.99      1.00      1.00   1649538



In [12]:
smote = SMOTE()
X_smoted, y_smoted = smote.fit_resample(X, y)

In [14]:
X_train_smote, X_test_smote, y_train_smote, y_test_smote = train_test_split(X_smoted, y_smoted, stratify=y_smoted)

In [16]:
n_features = X_smoted.shape[1]
X_train_smote_cnn = X_train_smote.to_numpy().reshape(-1, n_features, 1)
X_test_smtoe_cnn = X_test_smote.to_numpy().reshape(-1, n_features, 1)

In [24]:
model2 = Model(input_dimension=n_features, neurons=250, num_classes=1)
model2.build(input_shape=(None, n_features, 1))
model2.compile(loss=tf.keras.losses.BinaryCrossentropy(), optimizer='Adam', metrics=['accuracy'])

In [27]:
model2.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_4 (Conv1D)           multiple                  256       
                                                                 
 batch_normalization_6 (Batc  multiple                 256       
 hNormalization)                                                 
                                                                 
 max_pooling1d_4 (MaxPooling  multiple                 0         
 1D)                                                             
                                                                 
 conv1d_5 (Conv1D)           multiple                  24704     
                                                                 
 batch_normalization_7 (Batc  multiple                 512       
 hNormalization)                                                 
                                                           

In [ ]:
callbacks = [EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]
history = model2.fit(X_train_smote_cnn, y_train_smote, batch_size=100, epochs=20, verbose=2, validation_split=0.1, callbacks=callbacks)

Epoch 1/20
